In [1]:
import json
import time
import warnings
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sqlalchemy import create_engine, text
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# ID: (Latitude, Longitude)
# These are approximate stadium locations
TEAM_LOCATIONS = {
    1610612737: (33.7573, -84.3963),  # ATL (State Farm Arena)
    1610612738: (42.3662, -71.0621),  # BOS (TD Garden)
    1610612751: (40.6826, -73.9754),  # BKN (Barclays Center)
    1610612766: (35.2251, -80.8392),  # CHA (Spectrum Center)
    1610612741: (41.8807, -87.6742),  # CHI (United Center)
    1610612739: (41.4965, -81.6881),  # CLE (Rocket Mortgage FieldHouse)
    1610612742: (32.7905, -96.8103),  # DAL (American Airlines Center)
    1610612743: (39.7487, -105.0076),  # DEN (Ball Arena)
    1610612765: (42.3411, -83.0553),  # DET (Little Caesars Arena)
    1610612744: (37.7680, -122.3877),  # GSW (Chase Center)
    1610612745: (29.7508, -95.3621),  # HOU (Toyota Center)
    1610612754: (39.7640, -86.1555),  # IND (Gainbridge Fieldhouse)
    1610612746: (33.9425, -118.4081),  # LAC (Intuit Dome - broadly LA area)
    1610612747: (34.0430, -118.2673),  # LAL (Crypto.com Arena)
    1610612763: (35.1382, -90.0505),  # MEM (FedExForum)
    1610612748: (25.7814, -80.1870),  # MIA (Kaseya Center)
    1610612749: (43.0451, -87.9172),  # MIL (Fiserv Forum)
    1610612750: (44.9795, -93.2761),  # MIN (Target Center)
    1610612740: (29.9490, -90.0821),  # NOP (Smoothie King Center)
    1610612752: (40.7505, -73.9934),  # NYK (Madison Square Garden)
    1610612760: (35.4634, -97.5151),  # OKC (Paycom Center)
    1610612753: (28.5392, -81.3839),  # ORL (Kia Center)
    1610612755: (39.9012, -75.1720),  # PHI (Wells Fargo Center)
    1610612756: (33.4457, -112.0712),  # PHX (Footprint Center)
    1610612757: (45.5316, -122.6668),  # POR (Moda Center)
    1610612758: (38.5802, -121.4997),  # SAC (Golden 1 Center)
    1610612759: (29.4270, -98.4375),  # SAS (Frost Bank Center)
    1610612761: (43.6435, -79.3791),  # TOR (Scotiabank Arena)
    1610612762: (40.7683, -111.9011),  # UTA (Delta Center)
    1610612764: (38.8982, -77.0209),  # WAS (Capital One Arena)
}


class ModelTrainer:
    def __init__(self, engine, feature_results_path, random_state=42):
        self.engine = engine
        self.random_state = random_state

        with open(feature_results_path) as f:
            self.feature_results = json.load(f)

        # all of this is just getting the initial data frmo the feature selection results from the pipeline
        self.train_seasons_raw = self.feature_results["metadata"]["train_seasons"]
        self.val_seasons = self.feature_results["metadata"]["val_seasons"]
        self.test_seasons = self.feature_results["metadata"]["test_seasons"]

        self.calibrate_seasons = [self.train_seasons_raw[-1]]
        self.train_seasons = self.train_seasons_raw[:-1]

        self.excluded_seasons = ["22019", "22020"]

        self.optimal_n = self.feature_results["optimization"]["optimal_n_features"]
        self.features = self.feature_results["selection"]["selected_features"][: self.optimal_n]

    def _get_team_data(self):
        """Load team statistics from database."""
        with self.engine.connect() as conn:
            query = text("""
                WITH game_teams AS (
                    SELECT DISTINCT
                        tgs.game_id,
                        tgs.season_id,
                        tgs.team_game_date AS game_date,
                        MAX(CASE WHEN tgs.team_matchup LIKE '%vs.%' THEN tgs.team_id END) AS team1_id,
                        MAX(CASE WHEN tgs.team_matchup LIKE '%@%'   THEN tgs.team_id END) AS team2_id
                    FROM team_game_stats tgs
                    WHERE tgs.season_id NOT IN :excluded_seasons
                    GROUP BY tgs.game_id, tgs.season_id, tgs.team_game_date
                )
                SELECT
                    gt.game_id, gt.season_id, gt.game_date, gt.team1_id, gt.team2_id,

                    t1.avg_team_plus_minus AS team_plus_minus,
                    t1.avg_team_pts  AS team_pts,  t1.avg_team_fgm  AS team_fgm,
                    t1.avg_team_fga  AS team_fga,  t1.avg_team_fg_pct  AS team_fg_pct,
                    t1.avg_team_fg3m AS team_fg3m, t1.avg_team_fg3a AS team_fg3a,
                    t1.avg_team_fg3_pct AS team_fg3_pct,
                    t1.avg_team_ftm  AS team_ftm,  t1.avg_team_fta AS team_fta,
                    t1.avg_team_ft_pct AS team_ft_pct,
                    t1.avg_team_oreb AS team_oreb, t1.avg_team_dreb AS team_dreb,
                    t1.avg_team_reb  AS team_reb,  t1.avg_team_ast AS team_ast,
                    t1.avg_team_stl  AS team_stl,  t1.avg_team_blk AS team_blk,
                    t1.avg_team_tov  AS team_tov,  t1.avg_team_pf  AS team_pf,
                    t1.games_in_average AS team_games_in_avg,

                    t2.avg_team_plus_minus AS opp_plus_minus,
                    t2.avg_team_pts  AS opp_pts,  t2.avg_team_fgm  AS opp_fgm,
                    t2.avg_team_fga  AS opp_fga,  t2.avg_team_fg_pct  AS opp_fg_pct,
                    t2.avg_team_fg3m AS opp_fg3m, t2.avg_team_fg3a AS opp_fg3a,
                    t2.avg_team_fg3_pct AS opp_fg3_pct,
                    t2.avg_team_ftm  AS opp_ftm,  t2.avg_team_fta AS opp_fta,
                    t2.avg_team_ft_pct AS opp_ft_pct,
                    t2.avg_team_oreb AS opp_oreb, t2.avg_team_dreb AS opp_dreb,
                    t2.avg_team_reb  AS opp_reb,  t2.avg_team_ast AS opp_ast,
                    t2.avg_team_stl  AS opp_stl,  t2.avg_team_blk AS opp_blk,
                    t2.avg_team_tov  AS opp_tov,  t2.avg_team_pf  AS opp_pf,
                    t2.games_in_average AS opp_games_in_avg,
    
                    t1.avg_team_pts  - t2.avg_team_pts  AS diff_pts,
                    t1.avg_team_fgm  - t2.avg_team_fgm  AS diff_fgm,
                    t1.avg_team_fga  - t2.avg_team_fga  AS diff_fga,
                    t1.avg_team_fg_pct  - t2.avg_team_fg_pct  AS diff_fg_pct,
                    t1.avg_team_fg3m - t2.avg_team_fg3m AS diff_fg3m,
                    t1.avg_team_fg3a - t2.avg_team_fg3a AS diff_fg3a,
                    t1.avg_team_fg3_pct - t2.avg_team_fg3_pct AS diff_fg3_pct,
                    t1.avg_team_ftm  - t2.avg_team_ftm  AS diff_ftm,
                    t1.avg_team_fta  - t2.avg_team_fta  AS diff_fta,
                    t1.avg_team_ft_pct - t2.avg_team_ft_pct AS diff_ft_pct,
                    t1.avg_team_oreb - t2.avg_team_oreb AS diff_oreb,
                    t1.avg_team_dreb - t2.avg_team_dreb AS diff_dreb,
                    t1.avg_team_reb  - t2.avg_team_reb  AS diff_reb,
                    t1.avg_team_ast  - t2.avg_team_ast  AS diff_ast,
                    t1.avg_team_stl  - t2.avg_team_stl  AS diff_stl,
                    t1.avg_team_blk  - t2.avg_team_blk  AS diff_blk,
                    t1.avg_team_tov  - t2.avg_team_tov  AS diff_tov,
                    t1.avg_team_pf   - t2.avg_team_pf   AS diff_pf
                FROM game_teams gt
                JOIN team_average_game_stats t1
                    ON gt.game_id = t1.game_id AND gt.team1_id = t1.team_id
                JOIN team_average_game_stats t2
                    ON gt.game_id = t2.game_id AND gt.team2_id = t2.team_id
                WHERE t1.games_in_average >= 5
                  AND t2.games_in_average >= 5
                ORDER BY gt.game_date, gt.game_id;
            """)

            # return pd.read_sql(
            #     query, conn,
            #     params={'excluded_seasons': tuple(self.excluded_seasons)}
            # )
            # 1. Run the base SQL Query
            main_df = pd.read_sql(query, conn, params={"excluded_seasons": tuple(self.excluded_seasons)})

            # 2. Calculate Travel/Rest in Python
            print("   ✈️  Calculating travel logistics...")
            travel_lookup = self._get_travel_and_rest_data()

            # 3. Merge for Team 1 (Home usually)
            main_df = main_df.join(
                travel_lookup.rename(columns={"rest_days": "team_rest", "travel_dist": "team_dist"}),
                on=["game_id", "team1_id"],
            )

            # 4. Merge for Team 2 (Away usually)
            main_df = main_df.join(
                travel_lookup.rename(columns={"rest_days": "opp_rest", "travel_dist": "opp_dist"}),
                on=["game_id", "team2_id"],
            )

            # 5. Create Differential Features (Optional but recommended)
            main_df["diff_rest"] = main_df["team_rest"] - main_df["opp_rest"]
            main_df["diff_dist"] = main_df["team_dist"] - main_df["opp_dist"]

            return main_df.fillna(0)

    def _get_player_data(self):
        """Load player statistics from database."""
        with self.engine.connect() as conn:
            query = text("""
                WITH game_teams AS (
                    SELECT DISTINCT
                        tgs.game_id,
                        tgs.season_id,
                        MAX(CASE WHEN tgs.team_matchup LIKE '%vs.%' THEN tgs.team_id END) AS team1_id,
                        MAX(CASE WHEN tgs.team_matchup LIKE '%@%'   THEN tgs.team_id END) AS team2_id
                    FROM team_game_stats tgs
                    WHERE tgs.season_id NOT IN :excluded_seasons
                    GROUP BY tgs.game_id, tgs.season_id
                ),
                ranked_players AS (
                    SELECT
                        pgs.game_id, pgs.season_id, pgs.team_id, pgs.player_id,
                        p.player_name,
                        pgs.avg_min, pgs.avg_pts, pgs.avg_fgm, pgs.avg_fga,
                        pgs.avg_fg_pct, pgs.avg_fg3m, pgs.avg_fg3a, pgs.avg_fg3_pct,
                        pgs.avg_ftm, pgs.avg_fta, pgs.avg_ft_pct,
                        pgs.avg_oreb, pgs.avg_dreb, pgs.avg_reb,
                        pgs.avg_ast, pgs.avg_stl, pgs.avg_blk,
                        pgs.avg_tov, pgs.avg_pf, pgs.avg_plus_minus,
                        pgs.games_in_average,
                        ROW_NUMBER() OVER (
                            PARTITION BY pgs.game_id, pgs.team_id
                            ORDER BY pgs.avg_min DESC
                        ) AS player_rank
                    FROM player_average_game_stats pgs
                    JOIN players p ON pgs.player_id = p.player_id
                    WHERE pgs.avg_min > 0
                )
                SELECT
                    gt.game_id, gt.season_id,
                    'team' AS player_team_type,
                    gt.team1_id AS team_id,
                    rp.player_rank, rp.player_id, rp.player_name,
                    rp.avg_min, rp.avg_pts, rp.avg_fgm, rp.avg_fga,
                    rp.avg_fg_pct, rp.avg_fg3m, rp.avg_fg3a, rp.avg_fg3_pct,
                    rp.avg_ftm, rp.avg_fta, rp.avg_ft_pct,
                    rp.avg_oreb, rp.avg_dreb, rp.avg_reb,
                    rp.avg_ast, rp.avg_stl, rp.avg_blk,
                    rp.avg_tov, rp.avg_pf, rp.avg_plus_minus,
                    rp.games_in_average
                FROM game_teams gt
                JOIN ranked_players rp
                    ON gt.game_id = rp.game_id AND gt.team1_id = rp.team_id
                WHERE rp.player_rank <= 10
    
                UNION ALL
    
                SELECT
                    gt.game_id, gt.season_id,
                    'opp' AS player_team_type,
                    gt.team2_id AS team_id,
                    rp.player_rank, rp.player_id, rp.player_name,
                    rp.avg_min, rp.avg_pts, rp.avg_fgm, rp.avg_fga,
                    rp.avg_fg_pct, rp.avg_fg3m, rp.avg_fg3a, rp.avg_fg3_pct,
                    rp.avg_ftm, rp.avg_fta, rp.avg_ft_pct,
                    rp.avg_oreb, rp.avg_dreb, rp.avg_reb,
                    rp.avg_ast, rp.avg_stl, rp.avg_blk,
                    rp.avg_tov, rp.avg_pf, rp.avg_plus_minus,
                    rp.games_in_average
                FROM game_teams gt
                JOIN ranked_players rp
                    ON gt.game_id = rp.game_id AND gt.team2_id = rp.team_id
                WHERE rp.player_rank <= 10
    
                ORDER BY game_id, player_team_type, player_rank;
            """)

            return pd.read_sql(query, conn, params={"excluded_seasons": tuple(self.excluded_seasons)})

    def _calculate_sos(self, df):
        """
        Calculates cumulative Strength of Schedule (SOS) for both teams.
        Returns the original DataFrame with 'diff_sos' added.
        """
        print("   📊 Calculating Strength of Schedule (SOS)...")

        # 1. Create a "Long" view (Stack Home and Away games)
        # We need a single list of [Team, Date, OpponentStrength]
        home_view = df[["game_id", "game_date", "team1_id", "opp_plus_minus"]].rename(
            columns={"team1_id": "team_id", "opp_plus_minus": "opp_strength"}
        )
        away_view = df[["game_id", "game_date", "team2_id", "team_plus_minus"]].rename(
            columns={"team2_id": "team_id", "team_plus_minus": "opp_strength"}
        )

        schedule = pd.concat([home_view, away_view])

        # 2. Sort Chronologically by Team
        schedule["game_date"] = pd.to_datetime(schedule["game_date"])
        schedule = schedule.sort_values(["team_id", "game_date"])

        # 3. Calculate Expanding Mean (Shift 1 to exclude current game)
        schedule["sos"] = schedule.groupby("team_id")["opp_strength"].transform(lambda x: x.shift(1).expanding().mean())
        schedule["sos"] = schedule["sos"].fillna(0)

        # 4. Map back to Main DataFrame
        # Map Home Team's SOS
        df = df.merge(
            schedule[["game_id", "team_id", "sos"]].rename(columns={"team_id": "team1_id", "sos": "team1_sos"}),
            on=["game_id", "team1_id"],
            how="left",
        )
        # Map Away Team's SOS
        df = df.merge(
            schedule[["game_id", "team_id", "sos"]].rename(columns={"team_id": "team2_id", "sos": "team2_sos"}),
            on=["game_id", "team2_id"],
            how="left",
        )

        # 5. Create the Feature (Home SOS - Away SOS)
        df["diff_sos"] = df["team1_sos"] - df["team2_sos"]

        return df

    def _haversine(self, lat1, lon1, lat2, lon2):
        """Vectorized Haversine distance calculation."""
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        dlon = lon2 - lon1
        dlat = lat2 - lat1
        a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
        c = 2 * np.arcsin(np.sqrt(a))
        return c * 3956  # Miles

    def _get_travel_and_rest_data(self):
        """
        Fetches minimal schedule data to calculate rest and travel features.
        Returns a dictionary mapping (game_id, team_id) -> {rest, miles}
        """
        with self.engine.connect() as conn:
            # Get a simple schedule: Who played where and when?
            query = text("""
                SELECT 
                    team_id, game_id, team_game_date, team_matchup,
                    CASE WHEN team_matchup LIKE '%vs.%' THEN 1 ELSE 0 END as is_home
                FROM team_game_stats
                ORDER BY team_id, team_game_date
            """)
            df = pd.read_sql(query, conn)

        # 1. Setup Coordinates
        df["game_date"] = pd.to_datetime(df["team_game_date"])

        # Get My Location
        team_lat = {k: v[0] for k, v in TEAM_LOCATIONS.items()}
        team_lon = {k: v[1] for k, v in TEAM_LOCATIONS.items()}

        df["my_lat"] = df["team_id"].map(team_lat)
        df["my_lon"] = df["team_id"].map(team_lon)

        # Get Opponent Location (If I am away, I am at opp location. If home, at mine)
        # Note: Ideally we join to get opp_id, but for travel, "Away" usually implies
        # travel to the opponent. We can infer location if we had opponent ID.
        # CRITICAL FIX: To get precise travel, we need the opponent ID.
        # However, for a quick heuristic:
        # If Home: Location = My Stadium
        # If Away: Location = Opponent Stadium? We need Opponent ID.
        # Let's do a self-merge to get opponent ID easily.

        # Self-join to get opponent info (rows where same game_id, diff team_id)
        df_opp = df[["game_id", "team_id", "my_lat", "my_lon"]].rename(
            columns={"team_id": "opp_id", "my_lat": "opp_lat", "my_lon": "opp_lon"}
        )
        df = df.merge(df_opp, on="game_id")
        df = df[df["team_id"] != df["opp_id"]].copy()  # Remove self-matches

        # Determine Game Location
        df["loc_lat"] = np.where(df["is_home"] == 1, df["my_lat"], df["opp_lat"])
        df["loc_lon"] = np.where(df["is_home"] == 1, df["my_lon"], df["opp_lon"])

        # 2. Sort by Team sequence
        df = df.sort_values(["team_id", "game_date"])

        # 3. Calculate Lags (Previous Game info)
        df["prev_date"] = df.groupby("team_id")["game_date"].shift(1)
        df["prev_lat"] = df.groupby("team_id")["loc_lat"].shift(1)
        df["prev_lon"] = df.groupby("team_id")["loc_lon"].shift(1)

        # Fill First Game of Season (Assume Rest=7, Dist=0)
        df["prev_date"] = df["prev_date"].fillna(df["game_date"] - pd.Timedelta(days=7))
        df["prev_lat"] = df["prev_lat"].fillna(df["my_lat"])  # Start at home
        df["prev_lon"] = df["prev_lon"].fillna(df["my_lon"])

        # 4. Compute Metrics
        df["rest_days"] = (df["game_date"] - df["prev_date"]).dt.days
        df["rest_days"] = df["rest_days"].clip(0, 7)  # Cap at 7

        df["travel_dist"] = self._haversine(df["prev_lat"], df["prev_lon"], df["loc_lat"], df["loc_lon"])

        # Return as lookup table: Key=(game_id, team_id)
        return df.set_index(["game_id", "team_id"])[["rest_days", "travel_dist"]]

    def _merge_and_create_features(self, team_df, player_df):
        """
        Merge team and player data, pivot player stats.

        Args:
            team_df: Team statistics DataFrame
            player_df: Player statistics DataFrame

        Returns:
            DataFrame: ML-ready dataset
        """
        player_stats_cols = [
            "avg_min",
            "avg_pts",
            "avg_fgm",
            "avg_fga",
            "avg_fg_pct",
            "avg_fg3m",
            "avg_fg3a",
            "avg_fg3_pct",
            "avg_ftm",
            "avg_fta",
            "avg_ft_pct",
            "avg_oreb",
            "avg_dreb",
            "avg_reb",
            "avg_ast",
            "avg_stl",
            "avg_blk",
            "avg_tov",
            "avg_pf",
            "avg_plus_minus",
        ]

        # Efficient pivoting
        pivot = player_df.pivot_table(
            index="game_id",
            columns=["player_team_type", "player_rank"],
            values=player_stats_cols,
            aggfunc="first",
        )
        # Flatten multi-index columns: 'avg_pts', 'team', 1 -> 'team_pllayer1_avg_pts'
        pivot.columns = [f"{tt}_player{r}_{s}" for s, tt, r in pivot.columns]

        return team_df.merge(pivot, on="game_id", how="left")

    def _get_target_variable(self):  # Removed 'df' argument
        """Fetches final scores to create labels."""
        with self.engine.connect() as conn:
            query = text("""
                SELECT game_id,
                       CASE WHEN team_matchup LIKE '%vs.%' AND team_wl = 'W' THEN 1
                            WHEN team_matchup LIKE '%@%' AND team_wl = 'L' THEN 1
                            ELSE 0 END as target_win
                FROM team_game_stats
                WHERE season_id NOT IN :excluded_seasons
            """)
            target_df = pd.read_sql(query, conn, params={"excluded_seasons": tuple(self.excluded_seasons)})

            # We drop duplicates to ensure we have exactly one label per game_id
            return target_df.drop_duplicates("game_id")

    def load_data(self):
        """Load data and partition into 4 sets."""
        print("\n🔄 Loading data from database...")

        # 1. Load base data (COPY FROM PIPELINE)
        team_df = self._get_team_data()
        player_df = self._get_player_data()

        # 2. Merge and create features (COPY FROM PIPELINE)
        ml_df = self._merge_and_create_features(team_df, player_df)
        ml_df = self._calculate_sos(ml_df)
        ml_df = ml_df.fillna(0)

        # 3. Add target variable
        target_df = self._get_target_variable()
        ml_df = ml_df.merge(target_df, on="game_id", how="inner")

        ml_df["game_date"] = pd.to_datetime(ml_df["game_date"], errors="coerce")

        # Sort by Date, then ID. Reset index so row 0 is truly the first game.
        ml_df = ml_df.sort_values(["game_date", "game_id"]).reset_index(drop=True)

        # 4. PARTITION BY SEASON FIRST (before dropping season_id!)
        train_df = ml_df[ml_df["season_id"].isin(self.train_seasons)].copy()
        calibrate_df = ml_df[ml_df["season_id"].isin(self.calibrate_seasons)].copy()
        val_df = ml_df[ml_df["season_id"].isin(self.val_seasons)].copy()
        test_df = ml_df[ml_df["season_id"].isin(self.test_seasons)].copy()

        # 5. NOW drop ID columns from each split
        drop_cols = ["game_id", "season_id", "game_date", "team1_id", "team2_id"]
        train_df = train_df.drop(columns=drop_cols, errors="ignore")
        calibrate_df = calibrate_df.drop(columns=drop_cols, errors="ignore")
        val_df = val_df.drop(columns=drop_cols, errors="ignore")
        test_df = test_df.drop(columns=drop_cols, errors="ignore")

        # 6. Filter to selected features
        X_cols_filtered = [col for col in self.features if col in train_df.columns and col != "target_win"]
        y_col = "target_win"

        splits = {
            "train": train_df[X_cols_filtered + [y_col]],
            "calibrate": calibrate_df[X_cols_filtered + [y_col]],
            "val": val_df[X_cols_filtered + [y_col]],
            "test": test_df[X_cols_filtered + [y_col]],
        }
        print(f"   Train Rows:     {len(splits['train']):,}")
        print(f"   Calibrate Rows: {len(splits['calibrate']):,}")
        print(f"   Validate Rows:  {len(splits['val']):,}")
        print(f"   Test Rows:      {len(splits['test']):,}")

        return splits

    def _get_naive_baseline(self, y_train, y_compare, set_name="Validate"):
        # this will eventually be replaced with a bookmaker based baseline
        home_win_rate = y_train.mean()
        naive_preds = np.full(len(y_compare), home_win_rate)
        score = brier_score_loss(y_compare, naive_preds)

        print(f"\n📉 Naive Baseline ({set_name}):")
        print(f"   Strategy: Always predict {home_win_rate:.1%} (Train Win Rate)")
        print(f"   Brier Score: {score:.4f}")
        return score

    def _optimize_hyperparameters(self, X_train, y_train):
        start = time.time()

        param_dist = {
            "max_depth": [3, 4, 5, 6],
            "learning_rate": [0.01, 0.03, 0.05, 0.1],
            "min_child_weight": [1, 3, 5],
            "subsample": [0.6, 0.7, 0.8, 0.9],
            "colsample_bytree": [0.6, 0.7, 0.8, 0.9],
            "gamma": [0, 0.1, 0.2],
        }

        xgb = XGBClassifier(objective="binary:logistic", eval_metric="logloss", random_state=self.random_state)

        search = RandomizedSearchCV(
            xgb,
            param_distributions=param_dist,
            n_iter=20,
            scoring="neg_brier_score",
            cv=TimeSeriesSplit(n_splits=3),
            n_jobs=-1,
            random_state=self.random_state,
            verbose=1,
        )
        search.fit(X_train, y_train)

        print(f"   ✅ Best Params found in {time.time() - start:.1f}s:")
        print(f"      {search.best_params_}")

        return search.best_params_

    def _train_core_model(self, X_train, y_train, X_cal, y_cal, params=None, name="Default", early_stop_frac=0.1):
        # train XGBoost with Early Stopping on calibrate set
        print(f"Training {name} model...")

        split_idx = int(len(X_train) * (1 - early_stop_frac))
        X_fit, X_stop = X_train.iloc[:split_idx], X_train.iloc[split_idx:]
        y_fit, y_stop = y_train.iloc[:split_idx], y_train.iloc[split_idx:]

        # base model defaults if no params prodicer
        model_params = {
            "n_estimators": 2000,
            "max_depth": 4,
            "learning_rate": 0.03,
            "subsample": 0.8,
            "random_state": self.random_state,
            "eval_metric": "logloss",
            "early_stopping_rounds": 50,
        }

        if params:
            model_params.update(params)

        model = XGBClassifier(**model_params)

        model.fit(X_fit, y_fit, eval_set=[(X_stop, y_stop)], verbose=True)
        print(f"     Stopped at {model.best_iteration} trees")

        return model

    def _calculate_ece(self, y_true, y_prob, n_bins=10):
        # expected calibaertion error
        bins = np.linspace(0, 1, n_bins + 1)
        bin_ids = np.digitize(y_prob, bins) - 1

        bin_ids[bin_ids == n_bins] = n_bins - 1

        ece = 0.0
        total = len(y_true)

        for i in range(n_bins):
            mask = bin_ids == i
            n = mask.sum()
            if n > 0:
                avg_conf = y_prob[mask].mean()
                avg_acc = y_true[mask].mean()
                weight = n / total
                ece += weight * abs(avg_acc - avg_conf)

        return ece

    def _stratified_report(self, y_true, y_prob):
        # quANTILE BAED VALIBRATION REPORT (FAVS VS DOGS)
        df = pd.DataFrame({"prob": y_prob, "target": y_true})

        try:
            df["bucket"] = pd.qcut(df["prob"], q=5, duplicates="drop")
        except:
            df["bucket"] = pd.cut(df["prob"], bins=5)

        report = {}
        for interval, group in df.groupby("bucket", observed=False):
            n = len(group)
            if n > 0:
                pred = group["prob"].mean()
                obs = group["target"].mean()
                gap = obs - pred

                key = f"{interval.left:.2f}-{interval.right:.2f}"
                report[key] = {
                    "n": int(n),
                    "pred": float(pred),
                    "obs": float(obs),
                    "gap": float(gap),
                }
        return report

    def _evaluate(self, y_true, y_prob):
        # ccalucate ful metric suite
        return {
            "brier": brier_score_loss(y_true, y_prob),
            "ece": self._calculate_ece(y_true, y_prob),
            "log_loss": log_loss(y_true, y_prob),
            "auc": roc_auc_score(y_true, y_prob),
            "accuracy": accuracy_score(y_true, (y_prob > 0.5).astype(int)),
            "bias": float(np.mean(y_prob) - np.mean(y_true)),
            "stratified": self._stratified_report(y_true, y_prob),
        }

    def _print_evaluation(self, metrics, title):
        print(f"\n📊 {title} Results:")
        print(f"   Brier Score: {metrics['brier']:.4f}")
        print(f"   ECE:         {metrics['ece']:.4f}")
        print(f"   AUC:         {metrics['auc']:.4f}")
        print(f"   Bias:        {metrics['bias']:.4f} (Pos=Overconfident)")

        print("\n   🎯 Stratified Performance (Quantiles):")
        print(f"   {'Range':<14} | {'N':<5} | {'Pred':<6} | {'Obs':<6} | {'Gap':<6}")
        print("   " + "-" * 55)
        for rng, d in metrics["stratified"].items():
            print(f"   {rng:<14} | {d['n']:<5} | {d['pred']:.3f}  | {d['obs']:.3f}  | {d['gap']:+.3f}")

    def run_pipeline(self):
        # 1. Load Data
        data = self.load_data()
        # Features are already filtered in load_data; just separate X from y
        X_train = data["train"].drop(columns=["target_win"])
        y_train = data["train"]["target_win"]
        X_cal = data["calibrate"].drop(columns=["target_win"])
        y_cal = data["calibrate"]["target_win"]
        X_val = data["val"].drop(columns=["target_win"])
        y_val = data["val"]["target_win"]
        X_test = data["test"].drop(columns=["target_win"])
        y_test = data["test"]["target_win"]

        # 2. Establish Baseline
        val_baseline_brier = self._get_naive_baseline(y_train, y_val, "Validate")

        # 3. Train Models (Default vs Tuned)
        print("\n🏋️  Training Phase (Default vs. Tuned)")

        # A. Default
        model_default = self._train_core_model(X_train, y_train, X_cal, y_cal, name="Default")

        # B. Tuned
        best_params = self._optimize_hyperparameters(X_train, y_train)
        model_tuned = self._train_core_model(X_train, y_train, X_cal, y_cal, params=best_params, name="Tuned")

        # 4. Calibration & Selection (On Validation Set)
        print("\n🏆 Model Selection (Validated on held-out seasons)")

        candidates = []

        # Evaluate Default Variants
        for method in ["sigmoid", "isotonic"]:
            cal = CalibratedClassifierCV(model_default, method=method, cv="prefit")
            cal.fit(X_cal, y_cal)
            probs = cal.predict_proba(X_val)[:, 1]
            metrics = self._evaluate(y_val, probs)
            candidates.append({"name": f"Default_{method}", "model": cal, "metrics": metrics})

        # Evaluate Tuned Variants
        for method in ["sigmoid", "isotonic"]:
            cal = CalibratedClassifierCV(model_tuned, method=method, cv="prefit")
            cal.fit(X_cal, y_cal)
            probs = cal.predict_proba(X_val)[:, 1]
            metrics = self._evaluate(y_val, probs)
            candidates.append({"name": f"Tuned_{method}", "model": cal, "metrics": metrics})

        # Sort by Brier Score (Ascending)
        candidates.sort(key=lambda x: x["metrics"]["brier"])
        best_candidate = candidates[0]

        # Print Comparison
        print(f"\n   {'Model Name':<20} | {'Brier':<8} | {'ECE':<8} | {'AUC':<8}")
        print("   " + "-" * 55)
        for c in candidates:
            m = c["metrics"]
            print(f"   {c['name']:<20} | {m['brier']:.4f}   | {m['ece']:.4f}   | {m['auc']:.4f}")

        print(f"\n✨ WINNER: {best_candidate['name']}")
        improvement = val_baseline_brier - best_candidate["metrics"]["brier"]
        print(f"   Improvement over Naive: {improvement:.4f} ({(improvement / val_baseline_brier):.1%})")

        # 5. Final Test
        print("\n" + "=" * 80)
        print("FINAL AUDIT (Test Set)")
        print("=" * 80)

        final_probs = best_candidate["model"].predict_proba(X_test)[:, 1]
        test_metrics = self._evaluate(y_test, final_probs)
        self._print_evaluation(test_metrics, "Final Test Model")

        # Naive check for Test
        test_naive = self._get_naive_baseline(y_train, y_test, "Test")
        test_imp = test_naive - test_metrics["brier"]
        print(f"\n✅ Final Edge vs Naive: {test_imp:.4f}")

        return {
            "metadata": {
                "timestamp": datetime.now().isoformat(),
                "best_model": best_candidate["name"],
                "features_count": self.optimal_n,
                "features": list(X_train.columns),  # Store feature names for inference
            },
            "test_metrics": test_metrics,
            "best_params": best_params,
            "model": best_candidate["model"],
        }

    def save_model(self, model, filepath):
        """Save the trained model object."""

        joblib.dump(model, filepath)
        print(f"💾 Model saved to {filepath}")

    def load_model(self, filepath):
        """Load a previously trained model."""

        return joblib.load(filepath)

    def save_results(self, results, filepath):
        def convert(o):
            if isinstance(o, np.generic):
                return o.item()
            if isinstance(o, np.ndarray):
                return o.tolist()
            raise TypeError(f"Object of type {type(o)} is not JSON serializable")

        # Filter out non-serializable model object
        json_safe = {k: v for k, v in results.items() if k != "model"}

        with open(filepath, "w") as f:
            json.dump(json_safe, f, default=convert, indent=2)
        print(f"📄 Results saved to {filepath}")

    def save_all(self, results, model_path="nba_model.joblib", results_path="model_results.json"):
        """
        Convenience method to save both model and results.

        Args:
            results: Dictionary from run_pipeline()
            model_path: Path for model file
            results_path: Path for JSON metrics
        """
        self.save_model(results["model"], model_path)
        self.save_results(results, results_path)
        print("\n✅ All artifacts saved successfully")


# def main():
# from sqlalchemy import create_engine
# from dotenv import load_dotenv
# import os

# load_dotenv()
# DATABASE_URL = os.getenv("DATABASE_URL")
# engine = create_engine(DATABASE_URL)

# FEATURE_RESULTS_PATH = "feature_selection_results.json"

# engine = create_engine(DB_URL)
# trainer = ModelTrainer(engine, FEATURE_RESULTS_PATH)
# results = trainer.run_pipeline()
#     trainer.save_all(results, model_path=args.model_output, results_path=args.output)


# if __name__ == "__main__":
#     main()

In [8]:
import os

from dotenv import load_dotenv

load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

FEATURE_RESULTS_PATH = "feature_selection_results.json"

engine = create_engine(DATABASE_URL)
trainer = ModelTrainer(engine, FEATURE_RESULTS_PATH)
results = trainer.run_pipeline()
trainer.save_all(results, model_path="nba_model.joblib", results_path="model_results.json")


🔄 Loading data from database...
   Train Rows:     10,113
   Calibrate Rows: 1,151
   Validate Rows:  1,152
   Test Rows:      2,300

📉 Naive Baseline (Validate):
   Strategy: Always predict 58.8% (Train Win Rate)
   Brier Score: 0.2495

🏋️  Training Phase (Default vs. Tuned)
Training Default model...
[0]	validation_0-logloss:0.67754
[1]	validation_0-logloss:0.67472
[2]	validation_0-logloss:0.67171
[3]	validation_0-logloss:0.66942
[4]	validation_0-logloss:0.66700
[5]	validation_0-logloss:0.66489
[6]	validation_0-logloss:0.66283
[7]	validation_0-logloss:0.66106
[8]	validation_0-logloss:0.65947
[9]	validation_0-logloss:0.65765
[10]	validation_0-logloss:0.65599
[11]	validation_0-logloss:0.65439
[12]	validation_0-logloss:0.65288
[13]	validation_0-logloss:0.65135
[14]	validation_0-logloss:0.65030
[15]	validation_0-logloss:0.64913
[16]	validation_0-logloss:0.64799
[17]	validation_0-logloss:0.64703
[18]	validation_0-logloss:0.64589
[19]	validation_0-logloss:0.64500
[20]	validation_0-logloss: